In [1]:
# Se importan las librerías usadas para transformar fuentes operativas.

from pathlib import Path

import pandas as pd
import plotly.express as px

In [2]:
# Se definen las rutas de fuentes y del producto analítico permanente.

data_directory = Path("../data")
submission_directory = Path("../submission")
submission_directory.mkdir(exist_ok=True)

In [3]:
# Se cargan las líneas de pedido y se verifican su grano y campos disponibles.

order_lines = pd.read_csv(data_directory / "order_lines.csv")
order_lines.info()
order_lines.shape, order_lines[["order_id", "line_id"]].duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 471 entries, 0 to 470
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      471 non-null    int64  
 1   line_id       471 non-null    int64  
 2   order_date    471 non-null    object 
 3   customer_id   471 non-null    int64  
 4   product_id    471 non-null    int64  
 5   quantity      471 non-null    int64  
 6   discount_pct  471 non-null    float64
dtypes: float64(1), int64(5), object(1)
memory usage: 25.9+ KB


((471, 7), 0)

In [4]:
# Se cargan clientes y productos para incorporar atributos de análisis.

customers = pd.read_csv(data_directory / "customers.csv")
products = pd.read_csv(data_directory / "products.csv")
customers.shape, products.shape

((60, 3), (12, 4))

In [5]:
# Se integran las fuentes manteniendo una fila por línea de pedido.

sales_analytics = order_lines.merge(
    customers, on="customer_id", validate="many_to_one"
).merge(products, on="product_id", validate="many_to_one")
sales_analytics["order_date"] = pd.to_datetime(sales_analytics["order_date"])
sales_analytics["gross_sales"] = (
    sales_analytics["quantity"] * sales_analytics["unit_price"]
)
sales_analytics["discount_amount"] = (
    sales_analytics["gross_sales"] * sales_analytics["discount_pct"]
)
sales_analytics["net_sales"] = (
    sales_analytics["gross_sales"] - sales_analytics["discount_amount"]
)
sales_analytics.shape

(471, 15)

In [6]:
# Se verifica que la transformación preserve líneas, importes y relaciones válidas.

assert len(sales_analytics) == len(order_lines)
assert (
    sales_analytics[["customer_id", "product_id", "region", "category"]]
    .notna()
    .all()
    .all()
)
assert (
    sales_analytics["net_sales"].sum()
    == (sales_analytics["gross_sales"] - sales_analytics["discount_amount"]).sum()
)
sales_analytics[["order_id", "line_id", "region", "category", "net_sales"]].head()

,order_id,line_id,region,category,net_sales
0,1,1,Sur,Servicios,2327.5
1,1,2,Sur,Tecnología,2808.0
2,1,3,Sur,Tecnología,1020.0
3,2,1,Centro,Oficina,1140.0
4,3,1,Norte,Oficina,1305.0


In [7]:
# Se publica una tabla analítica con una fila por línea de pedido.

sales_analytics.to_csv(submission_directory / "sales_analytics.csv", index=False)
print("Grano publicado: una fila por línea de pedido.")

Grano publicado: una fila por línea de pedido.


In [8]:
# ¿Cómo cambian las ventas netas mensuales por categoría después de integrar las fuentes?

sales_analytics["month"] = (
    sales_analytics["order_date"].dt.to_period("M").dt.to_timestamp()
)
monthly_category = (
    sales_analytics.groupby(["month", "category"], as_index=False)["net_sales"]
    .sum()
    .sort_values(["month", "category"])
)
monthly_category

,month,category,net_sales
0,2024-01-01,Oficina,8481.5
1,2024-01-01,Servicios,19452.0
2,2024-01-01,Tecnología,22279.0
3,2024-02-01,Oficina,4548.0
4,2024-02-01,Servicios,29901.0
5,2024-02-01,Tecnología,16780.0
6,2024-03-01,Oficina,12045.5
7,2024-03-01,Servicios,23190.0
8,2024-03-01,Tecnología,22487.0
9,2024-04-01,Oficina,9028.5


In [9]:
# Se visualizan las ventas netas por categoría sin ocultar la dimensión temporal.

fig = px.line(
    monthly_category,
    x="month",
    y="net_sales",
    color="category",
    markers=True,
    title="Ventas netas mensuales por categoría",
    labels={"month": "Mes", "net_sales": "Ventas netas", "category": "Categoría"},
)
fig.update_layout(template="plotly_white")
fig.show()

In [10]:
import json

monthly_category.to_csv(
    submission_directory / "monthly_category_sales.csv", index=False
)
questions = [
    {
        "pregunta": "¿Cómo cambian las ventas netas mensuales por categoría después de integrar las fuentes?",
        "archivo_respuesta": "monthly_category_sales.csv",
    }
]
with (submission_directory / "questions.json").open("w", encoding="utf-8") as file:
    json.dump(questions, file, ensure_ascii=False, indent=2)